<a href="https://colab.research.google.com/github/DevzsJhonny/PROJECT_ONE_AI_AGENT/blob/main/Desenv_Agente_LOGICAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Instalação das bibliotecas
!pip install langchain pypdf sentence-transformers chromadb langchain-community

#!pip install google-generativeai PyPDF2 python-dotenv

from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Carregar o PDF
loader = PyPDFLoader("dados_logicar.pdf")
documentos = loader.load()


text_splitter = RecursiveCharacterTextSplitter (
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documentos)

print(f"Documento carregado com sucesso!")
print(f"Total de pedaços: {len(chunks)}")

Documento carregado com sucesso!
Total de pedaços: 2


In [3]:
#!pip install -U langchain-google-genai chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 14.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [10]:
import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [11]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA

minha_api_key = GEMINI_API_KEY

# 1. Configurar os Embeddings (essencial para o RAG encontrar os textos no PDF)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=minha_api_key
)

# 2. Criar o Banco de Dados com os 'chunks' do seu PDF da Soluções Logicar
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# 3. Configurar o LLM usando o modelo que funcionou no seu teste!
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    google_api_key=minha_api_key,
    temperature=0
)

# 4. Criar o Agente de Respostas
agente_logicar = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

print("✅ Agente Soluções Logicar ONLINE com o modelo Flash-Latest!")

✅ Agente Soluções Logicar ONLINE com o modelo Flash-Latest!


#Testando o Agente

In [12]:
pergunta = "De acordo com o manual, o que acontece se uma entrega de logística atrasar?"
try:
    resultado = agente_logicar.invoke(pergunta)
    print(f"RESPOSTA DO AGENTE: {resultado['result']}")
except Exception as e:
    print(f"Erro no teste: {e}")

RESPOSTA DO AGENTE: De acordo com o manual, caso uma entrega atrase mais de 4 horas além do prazo, o cliente recebe 10% de desconto no próximo frete.
